---
# **NBody com Python puro, Numpy e Cupy para 1D, 2D e 3D**
---

Estudante: André Destefani Stefanato

Orientador: Prof. Roberto Colistete Júnior

---
## **Bibliotecas importadas**
---

In [1]:
import numpy as np
import cupy as cp
import platform
import math

---
## **Informações sobre o hardware, o python e as bibliotecas usadas**
---

Nome do sistema operacional, nome do computador na rede, versão do kernel, arquitetura de hardware (32/64 bits), etc:

In [2]:
!uname -a

Linux 4646ed18aa7d 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux


Nome e versão do sistema operacional:

In [3]:
!lsb_release -a

No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 24.04.1 LTS
Release:	24.04
Codename:	noble


Diversas informações da CPU, como nome do processador, frequência em MHz da CPU, número de núcleos/cores, número de threads, memória cache, arquitetura de hardware (32/64 bits), etc:

In [4]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      12
  On-line CPU(s) list:       0-11
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   85
    Thread(s) per core:      2
    Core(s) per socket:      6
    Socket(s):               1
    Stepping:                7
    BogoMIPS:                4400.43
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                 

Partições do sistema de arquivos do sistema operacional:

In [5]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   66G  43% /
tmpfs            64M     0   64M   0% /dev
shm              41G     0   41G   0% /dev/shm
/dev/root       2.0G  1.3G  696M  65% /usr/sbin/docker-init
/dev/sda1       119G   53G   67G  44% /kaggle/input
tmpfs            42G  100K   42G   1% /var/colab
tmpfs            42G     0   42G   0% /proc/acpi
tmpfs            42G     0   42G   0% /proc/scsi
tmpfs            42G     0   42G   0% /sys/firmware


Memória RAM total e em uso pelo sistema operacional:

In [6]:
!free

               total        used        free      shared  buff/cache   available
Mem:        87528512     1937476    82289240        2296     4103084    85591036
Swap:              0           0           0


Versão do compilador C/C++ gcc:

In [7]:
!gcc --version

gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



Número da versão de Python:

In [8]:
platform.python_version()

'3.13.15'

data da versão:

In [9]:
platform.python_build()

('main', 'Aug  6 2026 11:06:22')

compilador C/C++ utilizado para criar tal versão de Python:

In [10]:
platform.python_compiler()

'GCC 13.3.0'

Versão de NumPy, onde 'np' é um apelido comum:

In [11]:
np.__version__

'2.1.3'

Versão de CuPy, onde 'cp' é um apelido comum:

In [12]:
cp.__version__

'14.0.1'

---
## **Inicialização do problema e fórmulas usadas**
---

Função de salvar snapshots

In [ ]:
def salvarSnapshot(df, biblioteca, dimensao):
  df.to_hdf("SnapshotTeste.h5", key=biblioteca + "/" + dimensao + "/Snapshots", mode="w", append=True, format='table' )

Variáveis usadas por todos os códigos

In [13]:
espacamento = 10
passos = 10
dt = 0.1
G = 1
epsilon = 5
numCorpos = 2**12
corposPorEixo3D = int(math.cbrt(numCorpos))
resto = numCorpos - corposPorEixo3D**3

Fórmulas da força gravitacional, energia cinética e energia potencial gravitacional respectivamente

$$ F = \sum_{\substack{j = 1 \\ j \ne i}}^{N} \frac{G m_i m_j}{|\vec{r}_{ij}|^2} $$

$$ K = \sum^N_{i=1} \frac{m_i |\vec{v}_i|^2}{2} $$

$$ U = \sum^N_{i=1}\sum_{\substack{j = 1 \\ j \ne i}}^{N} \frac{G m_i m_j}{|r_{ij}|} $$

---
## **Python puro**
---

### <font color="red">1D</font>

Inicialização de variáveis específicas da dimensão

In [ ]:
corposPorEixo = 2**12

x = []
v = [1.0] * numCorpos
m = [1.0] * numCorpos
forca = [0.0] * numCorpos
xTemp = 0

for i in range(corposPorEixo):
  xTemp += espacamento
  x.append(xTemp)

In [ ]:
def calculaForcas(numCorpos, x, m, G, epsilon, forca):
  for i in range(numCorpos):
    for j in range(i + 1, numCorpos):

      dx = x[j] - x[i]
      dist = (dx**2 + epsilon**2)

      invDist = dist**(-0.5)
      invDist = invDist * invDist * invDist

      forcaAtual = (G * m[i] * m[j]) * invDist

      forca[i] += forcaAtual * dx
      forca[j] += -forcaAtual * dx

In [ ]:
def movimentaCorpo(dt, v, x, m, numCorpos, G, epsilon, forca):
  for k in range(numCorpos):
    forca[k] = 0.0

  calculaForcas(numCorpos, x, m, G, epsilon, forca)
  for j in range(numCorpos):

    acel = forca[j] / m[j]

    v[j] = v[j] + acel * dt

    x[j] = x[j] + v[j] * dt

In [ ]:
def calculaEnergiaK(m, v, numCorpos):
  k = 0.0
  for i in range(numCorpos):
    k += 0.5 * m[i] * v[i]**2
  return k

In [ ]:
def calculaEnergiaU(numCorpos, x, m, G, epsilon):
  u = 0.0
  for i in range(numCorpos):
    for j in range(i + 1, numCorpos):
      dx = x[j] - x[i]
      dist = (dx**2 + epsilon**2)**0.5

      u += - (G * m[i] * m[j]) / dist

  return u

Resultados:

In [ ]:
%time k = calculaEnergiaK(m, v, numCorpos)
print(f"Energia Cinética antes da simulação: {k}")

CPU times: user 328 µs, sys: 0 ns, total: 328 µs
Wall time: 329 µs
Energia Cinética antes da simulação: 2048.0


In [ ]:
%time u = calculaEnergiaU(numCorpos, x, m, G, epsilon)
print(f"Energia Potencial antes da simulação: {u}")

CPU times: user 3.44 s, sys: 5.14 ms, total: 3.45 s
Wall time: 3.48 s
Energia Potencial antes da simulação: -3180.6036932616885


In [ ]:
snapshots = np.empty([passos * numCorpos, 8], dtype=np.float64)

In [ ]:
%%time
for i in range(passos):
    inicio = i * numCorpos
    fim = inicio + numCorpos
    snapshots[inicio:fim, 0:8] = np.array([x.copy(), np.full(numCorpos, 0), np.full(numCorpos, 0), v.copy(), np.full(numCorpos, 0),
                                             np.full(numCorpos, 0), m.copy(), np.full(numCorpos, i)]).transpose()

    movimentaCorpo(dt, v, x, m, numCorpos, G, epsilon, forca)

CPU times: user 47.9 s, sys: 28.5 ms, total: 48 s
Wall time: 48.3 s


In [ ]:
%time k = calculaEnergiaK(m, v, numCorpos)
print(f"Energia Cinética após a simulação: {k}")

CPU times: user 822 µs, sys: 0 ns, total: 822 µs
Wall time: 815 µs
Energia Cinética após a simulação: 2048.0005904357076


In [ ]:
%time u = calculaEnergiaU(numCorpos, x, m, G, epsilon)
print(f"Energia Potencial após a simulação: {u}")

CPU times: user 2.55 s, sys: 3.98 ms, total: 2.55 s
Wall time: 2.56 s
Energia Potencial após a simulação: -3180.6043230771256


### <font color="red">2D</font>

In [ ]:
corposPorEixo = 2**6

x = []
y = []
vx = [1.0] * numCorpos
vy = [1.0] * numCorpos
m = [1.0] * numCorpos
xTemp = 0

forcaX = [0.0] * numCorpos
forcaY = [0.0] * numCorpos

for i in range(corposPorEixo):
  xTemp += espacamento
  yTemp = 0
  for j in range(corposPorEixo):
    yTemp += espacamento
    x.append(xTemp)
    y.append(yTemp)

In [ ]:
def calculaForcas(numCorpos, x, y, m, G, epsilon, forcaX, forcaY):
  for i in range(numCorpos):
    for j in range(i + 1, numCorpos):
      dx = x[j] - x[i]
      dy = y[j] - y[i]

      dist = (dx**2 + dy**2 + epsilon**2)

      invDist = dist**(-0.5)
      invDist = invDist * invDist * invDist

      forcaAtual = (G * m[i] * m[j]) * invDist

      forcaX[i] += forcaAtual * dx
      forcaY[i] += forcaAtual * dy

      forcaX[j] += -forcaAtual * dx
      forcaY[j] += -forcaAtual * dy

In [ ]:
def movimentaCorpo(dt, vx, vy, x, y, m, numCorpos, G, epsilon, forcaX, forcaY):
  for k in range(numCorpos):
    forcaX[k] = 0.0
    forcaY[k] = 0.0

  calculaForcas(numCorpos, x, y, m, G, epsilon, forcaX, forcaY)
  for j in range(numCorpos):

    acelX = forcaX[j] / m[j]
    acelY = forcaY[j] / m[j]

    vx[j] = vx[j] + acelX * dt
    vy[j] = vy[j] + acelY * dt

    x[j] = x[j] + vx[j] * dt
    y[j] = y[j] + vy[j] * dt

In [ ]:
def calculaEnergiaK(m, vx, vy, numCorpos):
  k = 0.0
  v = [0.0] * numCorpos
  for i in range(numCorpos):
    v[i] = (vx[i]**2 + vy[i]**2)**0.5
    k += 0.5 * m[i] * v[i]**2
  return k

In [ ]:
def calculaEnergiaU(numCorpos, x, y, m, G, epsilon):
  u = 0.0
  for i in range(numCorpos):
    for j in range(i + 1, numCorpos):
      dx = x[j] - x[i]
      dy = y[j] - y[i]

      dist = (dx**2 + dy**2 + epsilon**2)**0.5

      u += - (G * m[i] * m[j]) / dist

  return u

Resultados:

In [ ]:
%time k = calculaEnergiaK(m, vx, vy, numCorpos)
print(f"Energia Cinética antes da simulação: {k}")

CPU times: user 1.12 ms, sys: 0 ns, total: 1.12 ms
Wall time: 1.12 ms
Energia Cinética antes da simulação: 4096.000000000001


In [ ]:
%time u = calculaEnergiaU(numCorpos, x, y, m, G, epsilon)
print(f"Energia Potencial antes da simulação: {u}")

CPU times: user 3.04 s, sys: 3 ms, total: 3.04 s
Wall time: 3.05 s
Energia Potencial antes da simulação: -37988.09648573182


In [ ]:
snapshots = np.empty([passos * numCorpos, 8], dtype=np.float64)

In [ ]:
%%time
for i in range(passos):
    inicio = i * numCorpos
    fim = inicio + numCorpos
    snapshots[inicio:fim, 0:8] = np.array([x.copy(), y.copy(), np.full(numCorpos, 0), vx.copy(), vy.copy(),
                                             np.full(numCorpos, 0), m.copy(), np.full(numCorpos, i)]).transpose()

    movimentaCorpo(dt, vx, vy, x, y, m, numCorpos, G, epsilon, forcaX, forcaY)

CPU times: user 1min 16s, sys: 44.8 ms, total: 1min 16s
Wall time: 1min 17s


In [ ]:
%time k = calculaEnergiaK(m, vx, vy, numCorpos)
print(f"Energia Cinética após a simulação: {k}")

CPU times: user 1.31 ms, sys: 1 µs, total: 1.31 ms
Wall time: 1.32 ms
Energia Cinética após a simulação: 4102.430013633144


In [ ]:
%time u = calculaEnergiaU(numCorpos, x, y, m, G, epsilon)
print(f"Energia Potencial após a simulação: {u}")

CPU times: user 3.07 s, sys: 5 ms, total: 3.08 s
Wall time: 3.08 s
Energia Potencial após a simulação: -37994.95530515816


### <font color="red">3D</font>

In [29]:
corposPorEixo = 2**4

x = []
y = []
z = []
vx = [1.0] * numCorpos
vy = [1.0] * numCorpos
vz = [1.0] * numCorpos
m = [1.0] * numCorpos
xTemp = 0

forcaX = [0.0] * numCorpos
forcaY = [0.0] * numCorpos
forcaZ = [0.0] * numCorpos

for i in range(corposPorEixo):
  xTemp += espacamento
  yTemp = 0
  for j in range(corposPorEixo):
    yTemp += espacamento
    zTemp = 0
    for k in range(corposPorEixo):
      zTemp += espacamento
      x.append(xTemp)
      y.append(yTemp)
      z.append(zTemp)

In [30]:
def calculaForcas(numCorpos, x, y, z, m, G, epsilon, forcaX, forcaY, forcaZ):
  for i in range(numCorpos):
    for j in range(i + 1, numCorpos):
      dx = x[j] - x[i]
      dy = y[j] - y[i]
      dz = z[j] - z[i]

      dist = (dx**2 + dy**2 + dz**2 + epsilon**2)
      invDist = dist**(-0.5)
      invDist = invDist * invDist * invDist

      forcaAtual = (G * m[i] * m[j]) * invDist

      forcaX[i] += forcaAtual * dx
      forcaY[i] += forcaAtual * dy
      forcaZ[i] += forcaAtual * dz

      forcaX[j] += -forcaAtual * dx
      forcaY[j] += -forcaAtual * dy
      forcaZ[j] += -forcaAtual * dz

In [31]:
def movimentaCorpo(dt, vx, vy, vz, x, y, z, m, numCorpos, G, epsilon, forcaX, forcaY, forcaZ):
  for k in range(numCorpos):
    forcaX[k] = 0.0
    forcaY[k] = 0.0
    forcaZ[k] = 0.0

  calculaForcas(numCorpos, x, y, z, m, G, epsilon, forcaX, forcaY, forcaZ)
  for j in range(numCorpos):

    acelX = forcaX[j] / m[j]
    acelY = forcaY[j] / m[j]
    acelZ = forcaZ[j] / m[j]

    vx[j] = vx[j] + acelX * dt
    vy[j] = vy[j] + acelY * dt
    vz[j] = vz[j] + acelZ * dt

    x[j] = x[j] + vx[j] * dt
    y[j] = y[j] + vy[j] * dt
    z[j] = z[j] + vz[j] * dt

In [32]:
def calculaEnergiaK(m, vx, vy, vz, numCorpos):
  k = 0.0
  v = [0.0] * numCorpos
  for i in range(numCorpos):
    v[i] = (vx[i]**2 + vy[i]**2 + vz[i]**2)**0.5
    k += 0.5 * m[i] * v[i]**2
  return k

In [33]:
def calculaEnergiaU(numCorpos, x, y, z, m, G, epsilon):
  u = 0.0
  for i in range(numCorpos):
    for j in range(i + 1, numCorpos):
      dx = x[j] - x[i]
      dy = y[j] - y[i]
      dz = z[j] - z[i]

      dist = (dx**2 + dy**2 + dz**2 + epsilon**2)**0.5

      u += - (G * m[i] * m[j]) / dist

  return u

Resultados:

In [34]:
%%time
for i in range(passos):
    movimentaCorpo(dt, vx, vy, vz, x, y, z, m, numCorpos, G, epsilon, forcaX, forcaY, forcaZ)

CPU times: user 1min 12s, sys: 2.4 ms, total: 1min 12s
Wall time: 1min 12s


---
## **Numpy**
---

### <font color="red">1D</font>

Inicialização dos atributos feita em Python puro, essa parte do código sera executada apenas uma vez. Os vetores gerados serão jogados para o Numpy depois

In [ ]:
N = 4096

m = [1] * (N)
vx = [1] * (N)
x = []

x_temp = 0

for i in range(N):
  x_temp += espacamento
  x.append(x_temp)

Aqui jogamos os arrays criados com Python puro para o Numpy

In [ ]:
np_x = np.array(x, dtype=np.float64)
np_m = np.array(m, dtype=np.float64)
np_vx = np.array(vx, dtype=np.float64)

Função de cálculo da força gravitacional de cada partícula dentro do sistema. A função retorna a força x total para cada partícula

In [ ]:
def calcularForcas(np_x, np_m, G, epsilon):
    dx = np_x[None, :] - np_x[:, None]

    dist = np.sqrt(dx**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    F = G * (np_m[:, None] * np_m[None, :]) / (dist**2)

    fx = F * (dx / dist)

    fx_total = np.sum(fx, axis=1)

    return fx_total

Função que usa o método integrador Euler-Cromer para simular a interação entre as partículas num intervalo de tempo atualizando as posições e velocidades x de cada partícula

In [ ]:
def atualizarPosicao(np_x, np_vx, np_m, dt, G, epsilon):
    fx = calcularForcas(np_x, np_m, G, epsilon)

    ax = fx / np_m

    np_vx += ax * dt

    np_x += np_vx * dt

Função que calcula a energia cinética total do sistema

In [ ]:
def calcularEnergiaCinetica(np_vx, np_m):
    return 0.5 * np.sum(np_m * (np_vx**2))

Função que calcula a energia potencial total do sistema

In [ ]:
def calcularEnergiaPotencial(np_x, np_m, G, epsilon):
    dx = np_x[None, :] - np_x[: ,None]

    dist = np.sqrt(dx**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    m_prod = np_m[:, None] * np_m[None, :]

    U = G * np.sum(m_prod / dist) / 2

    return U

---

Energia cinética total inicial do sistema

In [ ]:
%time K = calcularEnergiaCinetica(np_vx, np_m)
print(f"Energia cinética inicial do sistema: {K}")

CPU times: user 0 ns, sys: 691 µs, total: 691 µs
Wall time: 576 µs
Energia cinética inicial do sistema: 2048.0


Energia potencial total inicial do sistema

In [ ]:
%time U = calcularEnergiaPotencial(np_x, np_m, G, epsilon)
print(f"Energia potencial inicial do sistema: {U}")

CPU times: user 122 ms, sys: 125 ms, total: 247 ms
Wall time: 249 ms
Energia potencial inicial do sistema: 3180.6036932670736


Execução

In [ ]:
snapshots = np.empty([passos * numCorpos, 8], dtype=np.float64)

In [ ]:
%%time
for i in range(passos):
    inicio = i * numCorpos
    fim = inicio + numCorpos
    snapshots[inicio:fim, 0:8] = np.array([np_x.copy(), np.full(numCorpos, 0), np.full(numCorpos, 0), np_vx.copy(), np.full(numCorpos, 0),
                                             np.full(numCorpos, 0), np_m.copy(), np.full(numCorpos, i)]).transpose()

    atualizarPosicao(np_x, np_vx, np_m, dt, G, epsilon)

CPU times: user 2.99 s, sys: 2.08 s, total: 5.07 s
Wall time: 5.1 s


Energia cinética total final do sistema

In [ ]:
%time K = calcularEnergiaCinetica(np_vx, np_m)
print(f"Energia cinética final do sistema: {K}")

CPU times: user 851 µs, sys: 0 ns, total: 851 µs
Wall time: 1.45 ms
Energia cinética final do sistema: 2048.0005904357035


Energia potencial total final do sistema

In [ ]:
%time U = calcularEnergiaPotencial(np_x, np_m, G, epsilon)
print(f"Energia potencial final do sistema: {U}")

CPU times: user 168 ms, sys: 127 ms, total: 295 ms
Wall time: 311 ms
Energia potencial final do sistema: 3180.6043230779264


### <font color="red">2D</font>

Inicialização dos atributos feita em Python puro, essa parte do código sera executada apenas uma vez. Os vetores gerados serão jogados para o Numpy depois

In [ ]:
N = 64

m = [1] * (N**2)
vx = [1] * (N**2)
vy = [1] * (N**2)
x = []
y = []

x_temp = 0

for i in range(N):
  x_temp += espacamento
  y_temp = 0
  for j in range(N):
    y_temp += espacamento
    x.append(x_temp)
    y.append(y_temp)

Aqui jogamos os arrays criados com Python puro para o Numpy

In [ ]:
np_x = np.array(x, dtype=np.float64)
np_y = np.array(y, dtype=np.float64)
np_m = np.array(m, dtype=np.float64)
np_vx = np.array(vx, dtype=np.float64)
np_vy = np.array(vy, dtype=np.float64)

Função de cálculo da força gravitacional de cada partícula dentro do sistema. A função retorna a força x e y total para cada partícula

In [ ]:
def calcularForcas(np_x, np_y, np_m, G, epsilon):
    dx = np_x[None, :] - np_x[:, None]
    dy = np_y[None, :] - np_y[:, None]

    dist = np.sqrt(dx**2 + dy**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    F = G * (np_m[:, None] * np_m[None, :]) / (dist**2)

    fx = F * (dx / dist)
    fy = F * (dy / dist)

    fx_total = np.sum(fx, axis=1)
    fy_total = np.sum(fy, axis=1)

    return fx_total, fy_total

Usando o método integrador Euler-Cromer criamos a função que ira simular a interação entre as partículas num intervalo de tempo atualizando as posições e velocidades x e y de cada partícula

In [ ]:
def atualizarPosicao(np_x, np_y, np_vx, np_vy, np_m, dt, G, epsilon):
    fx, fy = calcularForcas(np_x, np_y, np_m, G, epsilon)

    ax = fx / np_m
    ay = fy / np_m

    np_vx += ax * dt
    np_vy += ay * dt

    np_x += np_vx * dt
    np_y += np_vy * dt

Função que calcula a energia cinética total do sistema

In [ ]:
def calcularEnergiaCinetica(np_vx, np_vy, np_m):
    return 0.5 * np.sum(np_m * (np_vx**2 + np_vy**2))

Função que calcula a energia potencial total do sistema

In [ ]:
def calcularEnergiaPotencial(np_x, np_y, np_m, G, epsilon):
    dx = np_x[None, :] - np_x[: ,None]
    dy = np_y[None, :] - np_y[: ,None]

    dist = np.sqrt(dx**2 + dy**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    m_prod = np_m[:, None] * np_m[None, :]

    U = G * np.sum(m_prod / dist) / 2

    return U

---

Energia cinética total inicial do sistema

In [ ]:
%time K = calcularEnergiaCinetica(np_vx, np_vy, np_m)
print(f"Energia cinética inicial do sistema: {K}")

CPU times: user 302 µs, sys: 64 µs, total: 366 µs
Wall time: 319 µs
Energia cinética inicial do sistema: 4096.0


Energia potencial total inicial do sistema

In [ ]:
%time U = calcularEnergiaPotencial(np_x, np_y, np_m, G, epsilon)
print(f"Energia potencial inicial do sistema: {U}")

CPU times: user 179 ms, sys: 172 ms, total: 351 ms
Wall time: 351 ms
Energia potencial inicial do sistema: 37988.09648594038


Execução

In [ ]:
snapshots = np.empty([passos * numCorpos, 8], dtype=np.float64)

In [ ]:
%%time
for i in range(passos):
    inicio = i * numCorpos
    fim = inicio + numCorpos
    snapshots[inicio:fim, 0:8] = np.array([np_x.copy(), np_y.copy(), np.full(numCorpos, 0), np_vx.copy(), np_vy.copy(),
                                             np.full(numCorpos, 0), np_m.copy(), np.full(numCorpos, i)]).transpose()

    atualizarPosicao(np_x, np_y, np_vx, np_vy, np_m, dt, G, epsilon)

CPU times: user 4.96 s, sys: 3.28 s, total: 8.24 s
Wall time: 8.3 s


Energia cinética total final do sistema

In [ ]:
%time K = calcularEnergiaCinetica(np_vx, np_vy, np_m)
print(f"Energia cinética final do sistema: {K}")

CPU times: user 1 ms, sys: 0 ns, total: 1 ms
Wall time: 1.4 ms
Energia cinética final do sistema: 4102.43001363314


Energia potencial total final do sistema

In [ ]:
%time U = calcularEnergiaPotencial(np_x, np_y, np_m, G, epsilon)
print(f"Energia potencial inicial do sistema: {U}")

CPU times: user 206 ms, sys: 175 ms, total: 381 ms
Wall time: 451 ms
Energia potencial inicial do sistema: 37994.955305153715


### <font color="red">3D</font>

Inicialização dos atributos feita em Python puro, essa parte do código sera executada apenas uma vez. Os vetores gerados serão jogados para o Numpy depois

In [22]:
x = []
y = []
z = []
vx = [1.0] * numCorpos
vy = [1.0] * numCorpos
vz = [1.0] * numCorpos
m = [1.0] * numCorpos
xTemp = 0
yTemp = 0
zTemp = 0

for i in range(corposPorEixo3D):
  xTemp += espacamento
  yTemp = 0
  for j in range(corposPorEixo3D):
    yTemp += espacamento
    zTemp = 0
    for k in range(corposPorEixo3D):
      zTemp += espacamento
      x.append(xTemp)
      y.append(yTemp)
      z.append(zTemp)

if (resto != 0):
    for i in range (resto):
        xTemp += espacamento
        x.append(xTemp)
        yTemp += espacamento
        y.append(yTemp)
        zTemp += espacamento
        z.append(zTemp)

Aqui jogamos os arrays criados com Python puro para o Numpy

In [23]:
np_x = np.array(x, dtype=np.float64)
np_y = np.array(y, dtype=np.float64)
np_z = np.array(z, dtype=np.float64)
np_m = np.array(m, dtype=np.float64)
np_vx = np.array(vx, dtype=np.float64)
np_vy = np.array(vy, dtype=np.float64)
np_vz = np.array(vz, dtype=np.float64)

Função de cálculo da força gravitacional de cada partícula dentro do sistema. A função retorna a força x, y e z total para cada partícula

In [24]:
def calcularForcas(np_x, np_y, np_z, np_m, G, epsilon):
    dx = np_x[None, :] - np_x[:, None]
    dy = np_y[None, :] - np_y[:, None]
    dz = np_z[None, :] - np_z[:, None]

    dist = np.sqrt(dx**2 + dy**2 + dz**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    F = G * (np_m[:, None] * np_m[None, :]) / (dist**2)

    fx = F * (dx / dist)
    fy = F * (dy / dist)
    fz = F * (dz / dist)

    fx_total = np.sum(fx, axis=1)
    fy_total = np.sum(fy, axis=1)
    fz_total = np.sum(fz, axis=1)

    return fx_total, fy_total, fz_total

Usando o método integrador Euler-Cromer criamos a função que ira simular a interação entre as partículas num intervalo de tempo atualizando as posições e velocidades x, y e z de cada partícula

In [25]:
def atualizarPosicao(np_x, np_y, np_z, np_vx, np_vy, np_vz, np_m, dt, G, epsilon):
    fx, fy, fz = calcularForcas(np_x, np_y, np_z, np_m, G, epsilon)

    ax = fx / np_m
    ay = fy / np_m
    az = fz / np_m

    np_vx += ax * dt
    np_vy += ay * dt
    np_vz += az * dt

    np_x += np_vx * dt
    np_y += np_vy * dt
    np_z += np_vz * dt

Função que calcula a energia cinética total do sistema

In [26]:
def calcularEnergiaCinetica(np_vx, np_vy, np_vz, np_m):
    return 0.5 * np.sum(np_m * (np_vx**2 + np_vy**2 + np_vz**2))

Função que calcula a energia potencial total do sistema

In [27]:
def calcularEnergiaPotencial(np_x, np_y, np_z, np_m, G, epsilon):
    dx = np_x[None, :] - np_x[: ,None]
    dy = np_y[None, :] - np_y[: ,None]
    dz = np_z[None, :] - np_z[: ,None]

    dist = np.sqrt(dx**2 + dy**2 + dz**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    m_prod = np_m[:, None] * np_m[None, :]

    U = G * np.sum(m_prod / dist) / 2

    return U

---

Execução

In [28]:
%%time
for i in range(passos):
    atualizarPosicao(np_x, np_y, np_z, np_vx, np_vy, np_vz, np_m, dt, G, epsilon)

CPU times: user 4.8 s, sys: 4.73 s, total: 9.53 s
Wall time: 9.53 s


---
## **Cupy**
---

### <font color="red">1D</font>

Inicialização dos atributos feita em Python puro, essa parte do código sera executada apenas uma vez. Os vetores gerados serão jogados para o Cupy depois

In [ ]:
N = 4096

m = [1] * (N)
vx = [1] * (N)
x = []

x_temp = 0

for i in range(N):
  x_temp += espacamento
  x.append(x_temp)

Aqui jogamos os arrays criados com Python puro para o Cupy

In [ ]:
cp_x = cp.array(x, dtype=cp.float64)
cp_m = cp.array(m, dtype=cp.float64)
cp_vx = cp.array(vx, dtype=cp.float64)

Função de cálculo da força gravitacional de cada partícula dentro do sistema. A função retorna a força x total para cada partícula

In [ ]:
def calcularForcas(cp_x, cp_m, G, epsilon):
    dx = cp_x[None, :] - cp_x[:, None]

    dist = cp.sqrt(dx**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    F = G * (cp_m[:, None] * cp_m[None, :]) / (dist**2)

    fx = F * (dx / dist)

    fx_total = cp.sum(fx, axis=1)

    return fx_total

Função que usa o método integrador Euler-Cromer para simular a interação entre as partículas num intervalo de tempo atualizando as posições e velocidades x de cada partícula

In [ ]:
def atualizarPosicao(cp_x, cp_vx, cp_m, dt, G, epsilon):
    fx = calcularForcas(cp_x, cp_m, G, epsilon)

    ax = fx / cp_m

    cp_vx += ax * dt

    cp_x += cp_vx * dt

Função que calcula a energia cinética total do sistema

In [ ]:
def calcularEnergiaCinetica(cp_vx, cp_m):
    return 0.5 * cp.sum(cp_m * (cp_vx**2))

Função que calcula a energia potencial total do sistema

In [ ]:
def calcularEnergiaPotencial(cp_x, cp_m, G, epsilon):
    dx = cp_x[None, :] - cp_x[: ,None]

    dist = cp.sqrt(dx**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    m_prod = cp_m[:, None] * cp_m[None, :]

    U = G * cp.sum(m_prod / dist) / 2

    return U

---

Energia cinética total inicial do sistema

In [ ]:
%time K = calcularEnergiaCinetica(cp_vx, cp_m)
print(f"Energia cinética inicial do sistema: {K}")

CPU times: user 424 µs, sys: 0 ns, total: 424 µs
Wall time: 431 µs
Energia cinética inicial do sistema: 2048.0


Enegia potencial total inicial do sistema

In [ ]:
%time U = calcularEnergiaPotencial(cp_x, cp_m, G, epsilon)
print(f"Energia potencial inicial do sistema: {U}")

CPU times: user 1.56 ms, sys: 286 µs, total: 1.84 ms
Wall time: 2.69 ms
Energia potencial inicial do sistema: 3180.6036932670727


Execução

In [ ]:
snapshots = cp.empty([passos * numCorpos, 8], dtype=cp.float64)

In [ ]:
%%time
for i in range(passos):
    inicio = i * numCorpos
    fim = inicio + numCorpos
    snapshots[inicio:fim, 0:8] = cp.array([cp_x.copy(), cp.full(numCorpos, 0), cp.full(numCorpos, 0), cp_vx.copy(), cp.full(numCorpos, 0),
                                             cp.full(numCorpos, 0), cp_m.copy(), cp.full(numCorpos, i)]).transpose()

    atualizarPosicao(cp_x, cp_vx, cp_m, dt, G, epsilon)

CPU times: user 15.9 ms, sys: 87 µs, total: 16 ms
Wall time: 16.3 ms


Energia cinética total final do sistema

In [ ]:
%time K = calcularEnergiaCinetica(cp_vx, cp_m)
print(f"Energia cinética final do sistema: {K}")

CPU times: user 964 µs, sys: 0 ns, total: 964 µs
Wall time: 1.12 ms
Energia cinética final do sistema: 2048.000590435704


Energia potencial total final do sistema

In [ ]:
%time U = calcularEnergiaPotencial(cp_x, cp_m, G, epsilon)
print(f"Energia potencial inicial do sistema: {U}")

CPU times: user 606 µs, sys: 800 µs, total: 1.41 ms
Wall time: 1.35 ms
Energia potencial inicial do sistema: 3180.604323077931


### <font color="red">2D</font>

Inicialização dos atributos feita em Python puro, essa parte do código sera executada apenas uma vez. Os vetores gerados serão jogados para o Cupy depois

In [ ]:
N = 64

m = [1] * (N**2)
vx = [1] * (N**2)
vy = [1] * (N**2)
x = []
y = []

x_temp = 0

for i in range(N):
  x_temp += espacamento
  y_temp = 0
  for j in range(N):
    y_temp += espacamento
    x.append(x_temp)
    y.append(y_temp)

Aqui jogamos os arrays criados com Python puro para o Cupy

In [ ]:
cp_x = cp.array(x, dtype=cp.float64)
cp_y = cp.array(y, dtype=cp.float64)
cp_m = cp.array(m, dtype=cp.float64)
cp_vx = cp.array(vx, dtype=cp.float64)
cp_vy = cp.array(vy, dtype=cp.float64)

Função de cálculo da força gravitacional de cada partícula dentro do sistema. A função retorna a força x e y total para cada partícula

In [ ]:
def calcularForcas(cp_x, cp_y, cp_m, G, epsilon):
    dx = cp_x[None, :] - cp_x[:, None]
    dy = cp_y[None, :] - cp_y[:, None]

    dist = cp.sqrt(dx**2 + dy**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    F = G * (cp_m[:, None] * cp_m[None, :]) / (dist**2)

    fx = F * (dx / dist)
    fy = F * (dy / dist)

    fx_total = cp.sum(fx, axis=1)
    fy_total = cp.sum(fy, axis=1)

    return fx_total, fy_total

Usando o método integrador Euler-Cromer criamos a função que ira simular a interação entre as partículas num intervalo de tempo atualizando as posições e velocidades x e y de cada partícula

In [ ]:
def atualizarPosicao(cp_x, cp_y, cp_vx, cp_vy, cp_m, dt, G, epsilon):
    fx, fy = calcularForcas(cp_x, cp_y, cp_m, G, epsilon)

    ax = fx / cp_m
    ay = fy / cp_m

    cp_vx += ax * dt
    cp_vy += ay * dt

    cp_x += cp_vx * dt
    cp_y += cp_vy * dt

Função que calcula a energia cinética total do sistema

In [ ]:
def calcularEnergiaCinetica(cp_vx, cp_vy, cp_m):
    return 0.5 * cp.sum(cp_m * (cp_vx**2 + cp_vy**2))

Função que calcula a energia potencial total do sistema

In [ ]:
def calcularEnergiaPotencial(cp_x, cp_y, cp_m, G, epsilon):
    dx = cp_x[None, :] - cp_x[: ,None]
    dy = cp_y[None, :] - cp_y[: ,None]

    dist = cp.sqrt(dx**2 + dy**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    m_prod = cp_m[:, None] * cp_m[None, :]

    U = G * cp.sum(m_prod / dist) / 2

    return U

---

Energia cinética total inicial do sistema

In [ ]:
%time K = calcularEnergiaCinetica(cp_vx, cp_vy, cp_m)
print(f"Energia cinética inicial do sistema: {K}")

CPU times: user 1.77 ms, sys: 0 ns, total: 1.77 ms
Wall time: 1.52 ms
Energia cinética inicial do sistema: 4096.0


Energia potencial total inicial do sistema

In [ ]:
%time U = calcularEnergiaPotencial(cp_x, cp_y, cp_m, G, epsilon)
print(f"Energia potencial inicial do sistema: {U}")

CPU times: user 1.71 ms, sys: 0 ns, total: 1.71 ms
Wall time: 1.54 ms
Energia potencial inicial do sistema: 37988.09648594025


Execução

In [ ]:
snapshots = cp.empty([passos * numCorpos, 8], dtype=cp.float64)

In [ ]:
%%time
for i in range(passos):
    inicio = i * numCorpos
    fim = inicio + numCorpos
    snapshots[inicio:fim, 0:8] = cp.array([cp_x.copy(), cp_y.copy(), cp.full(numCorpos, 0), cp_vx.copy(), cp_vy.copy(),
                                             cp.full(numCorpos, 0), cp_m.copy(), cp.full(numCorpos, i)]).transpose()

    atualizarPosicao(cp_x, cp_y, cp_vx, cp_vy, cp_m, dt, G, epsilon)

CPU times: user 16.3 ms, sys: 1.06 ms, total: 17.4 ms
Wall time: 17.3 ms


Energia cinética total final do sistema

In [ ]:
%time K = calcularEnergiaCinetica(cp_vx, cp_vy, cp_m)
print(f"Energia cinética final do sistema: {K}")

CPU times: user 1.24 ms, sys: 119 µs, total: 1.36 ms
Wall time: 1.17 ms
Energia cinética final do sistema: 4102.43001363314


Energia potencial total final do sistema

In [ ]:
%time U = calcularEnergiaPotencial(cp_x, cp_y, cp_m, G, epsilon)
print(f"Energia potencial inicial do sistema: {U}")

CPU times: user 1.52 ms, sys: 11 µs, total: 1.53 ms
Wall time: 1.38 ms
Energia potencial inicial do sistema: 37994.955305153744


### <font color="red">3D</font>

Inicialização dos atributos feita em Python puro, essa parte do código sera executada apenas uma vez. Os vetores gerados serão jogados para o Cupy depois

In [14]:
x = []
y = []
z = []
vx = [1.0] * numCorpos
vy = [1.0] * numCorpos
vz = [1.0] * numCorpos
m = [1.0] * numCorpos
xTemp = 0
yTemp = 0
zTemp = 0

for i in range(corposPorEixo3D):
  xTemp += espacamento
  yTemp = 0
  for j in range(corposPorEixo3D):
    yTemp += espacamento
    zTemp = 0
    for k in range(corposPorEixo3D):
      zTemp += espacamento
      x.append(xTemp)
      y.append(yTemp)
      z.append(zTemp)

if (resto != 0):
    for i in range (resto):
        xTemp += espacamento
        x.append(xTemp)
        yTemp += espacamento
        y.append(yTemp)
        zTemp += espacamento
        z.append(zTemp)

Aqui jogamos os arrays criados com Python puro para o Cupy

In [15]:
cp_x = cp.array(x, dtype=cp.float64)
cp_y = cp.array(y, dtype=cp.float64)
cp_z = cp.array(z, dtype=cp.float64)
cp_m = cp.array(m, dtype=cp.float64)
cp_vx = cp.array(vx, dtype=cp.float64)
cp_vy = cp.array(vy, dtype=cp.float64)
cp_vz = cp.array(vz, dtype=cp.float64)

Função de cálculo da força gravitacional de cada partícula dentro do sistema. A função retorna a força x, y e z total para cada partícula

In [16]:
def calcularForcas(cp_x, cp_y, cp_z, cp_m, G, epsilon):
    dx = cp_x[None, :] - cp_x[:, None]
    dy = cp_y[None, :] - cp_y[:, None]
    dz = cp_z[None, :] - cp_z[:, None]

    dist = cp.sqrt(dx**2 + dy**2 + dz**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    F = G * (cp_m[:, None] * cp_m[None, :]) / (dist**2)

    fx = F * (dx / dist)
    fy = F * (dy / dist)
    fz = F * (dz / dist)

    fx_total = cp.sum(fx, axis=1)
    fy_total = cp.sum(fy, axis=1)
    fz_total = cp.sum(fz, axis=1)

    return fx_total, fy_total, fz_total

Usando o método integrador Euler-Cromer criamos a função que ira simular a interação entre as partículas num intervalo de tempo atualizando as posições e velocidades x, y e z de cada partícula

In [17]:
def atualizarPosicao(cp_x, cp_y, cp_z, cp_vx, cp_vy, cp_vz, cp_m, dt, G, epsilon):
    fx, fy, fz = calcularForcas(cp_x, cp_y, cp_z, cp_m, G, epsilon)

    ax = fx / cp_m
    ay = fy / cp_m
    az = fz / cp_m

    cp_vx += ax * dt
    cp_vy += ay * dt
    cp_vz += az * dt

    cp_x += cp_vx * dt
    cp_y += cp_vy * dt
    cp_z += cp_vz * dt

Função que calcula a energia cinética total do sistema

In [18]:
def calcularEnergiaCinetica(cp_vx, cp_vy, cp_vz, cp_m):
    return 0.5 * cp.sum(cp_m * (cp_vx**2 + cp_vy**2 + cp_vz**2))

Função que calcula a energia potencial total do sistema

In [19]:
def calcularEnergiaPotencial(cp_x, cp_y, cp_z, cp_m, G, epsilon):
    dx = cp_x[None, :] - cp_x[: ,None]
    dy = cp_y[None, :] - cp_y[: ,None]
    dz = cp_z[None, :] - cp_z[: ,None]

    dist = cp.sqrt(dx**2 + dy**2 + dz**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    m_prod = cp_m[:, None] * cp_m[None, :]

    U = G * cp.sum(m_prod / dist) / 2

    return U

---

In [21]:
%%time
for i in range(passos):
    atualizarPosicao(cp_x, cp_y, cp_z, cp_vx, cp_vy, cp_vz, cp_m, dt, G, epsilon)

CPU times: user 1.45 s, sys: 90.7 ms, total: 1.54 s
Wall time: 1.76 s
